In [41]:
import os 
os.chdir('C:/Users/user/OneDrive/Desktop/MyProjects/Project2')

In [42]:
import fastf1
import pandas as pd
fastf1.Cache.enable_cache('C:/Users/user/OneDrive/Desktop/MyProjects/Project2/cache') #enabling cache to speed up data retrieval
session = fastf1.get_session(2024, 'Japan', 'R') #getting the race session for the 2024 Japanese Grand Prix
session.load() #loading the session data
laps = session.laps.copy() #copying the laps data to a new DataFrame for manipulation
laps['LapTimeSeconds'] = laps["LapTime"].dt.total_seconds() #converting lap times to seconds for easier analysis
print("Total laps before cleaning:", len(laps))

core           INFO 	Loading data for Japanese Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '11', '55', '16', '4', '14', '63', '81', '44', '22', '27', '18', '20', '77', '31', '10', '2', '24', '3', '23']


Total laps before cleaning: 907


In [ ]:
cleaned_laps = laps.dropna(subset=['LapTime']) #removing laps with missing data (no time recorded, likely due to retirements, disqualifications, or other issues))
print("Total laps after cleaning:", len(cleaned_laps))

Total laps after cleaning: 876


In [ ]:
cleaned_laps = cleaned_laps[cleaned_laps['PitInTime'].isna()] #removing laps with missing pit stop data (slow due to pit entry)
print("Total laps after removing missing pit stop data:", len(cleaned_laps))

Total laps after removing missing pit stop data: 821


In [ ]:
cleaned_laps = cleaned_laps[cleaned_laps['PitOutTime'].isna()] #removing laps with missing pit stop data (out-laps where the car pitted but the out time is missing would also be incomplete)
print("Total laps after removing missing pit stop data:", len(cleaned_laps))

Total laps after removing missing pit stop data: 786


In [46]:
cleaned_laps = cleaned_laps[cleaned_laps['Compound'].isin(['SOFT', 'MEDIUM', 'HARD'])] #removing laps with invalid tyre compound data
print("Total laps after removing invalid tyre compound data:", len(cleaned_laps))

Total laps after removing invalid tyre compound data: 786


In [ ]:
def remove_outliers(driver_laps):
    median = driver_laps['LapTimeSeconds'].median()
    return driver_laps[(driver_laps['LapTimeSeconds'] <= 1.10 * median)]

cleaned_laps = cleaned_laps.groupby('Driver').apply(remove_outliers) #removing outliers for each driver, safety cars, crashes, and other anomalies can cause unusually slow lap times that would skew the analysis
print ("Total laps after removing outliers:", len(cleaned_laps))

Total laps after removing outliers: 779


C:\Users\user\AppData\Local\Temp\ipykernel_16216\1404817007.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  cleaned_laps = cleaned_laps.groupby('Driver').apply(remove_outliers) #removing outliers for each driver


In [48]:
cleaned_laps[['Driver', 'Team', 'LapNumber', 'LapTimeSeconds', 'Compound', 'TyreLife', 'Stint', 'Position']].to_csv('C:/Users/user/OneDrive/Desktop/MyProjects/Project2/data/japan_2024_clean.csv', index=False)
print("Cleaned data saved to CSV file.")

Cleaned data saved to CSV file.


In [50]:
verify = pd.read_csv('C:/Users/user/OneDrive/Desktop/MyProjects/Project2/data/japan_2024_clean.csv')
print('Reload check:', verify.shape)
print("Drivers in cleaned data:", verify['Driver'].unique())

Reload check: (779, 8)
Drivers in cleaned data: ['ALO' 'BOT' 'GAS' 'HAM' 'HUL' 'LEC' 'MAG' 'NOR' 'OCO' 'PER' 'PIA' 'RUS'
 'SAI' 'SAR' 'STR' 'TSU' 'VER' 'ZHO']
